<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/Groq_MultiAgentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# INSTALAR DEPENDÊNCIA
# ==========================================
!pip install -q groq

# ==========================================
# IMPORTS
# ==========================================
from groq import Groq
from google.colab import userdata
import time

# ==========================================
# CONFIG - MODELOS ATUALIZADOS
# ==========================================
MODELOS = [
    "llama-3.3-70b-versatile",  # principal (mais atual)
    "llama-3.3-8b-instant",     # rápido
    "mixtral-8x7b-32768"        # fallback estável
]

# ==========================================
# API KEY (SECRETS DO COLAB)
# ==========================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("❌ API Key não encontrada nos Secrets (GROQ_API_KEY)")

client = Groq(api_key=GROQ_API_KEY)

print("✅ API conectada com sucesso!")

# ==========================================
# CRIAÇÃO DE AGENTES
# ==========================================
def criar_agente(nome, personalidade):
    return {
        "nome": nome,
        "personalidade": personalidade,
        "historico": []
    }

agente_1 = criar_agente(
    "Atendente",
    """
    Você é um atendente profissional, educado e eficiente.
    Resolve problemas com clareza e objetividade.
    Nunca seja grosseiro.
    """
)

agente_2 = criar_agente(
    "Cliente",
    """
    Você é um cliente com problema.
    Pode demonstrar insatisfação leve ou moderada.
    Quer solução rápida e clara.
    """
)

# ==========================================
# FUNÇÃO DE RESPOSTA COM FALLBACK INTELIGENTE
# ==========================================
def gerar_resposta(mensagem, agente):
    ultimo_erro = None

    for modelo in MODELOS:
        try:
            resposta = client.chat.completions.create(
                model=modelo,
                messages=[
                    {"role": "system", "content": agente["personalidade"]},
                    *agente["historico"],
                    {"role": "user", "content": mensagem}
                ],
                temperature=0.7,
            )

            texto = resposta.choices[0].message.content

            # salvar histórico
            agente["historico"].append({"role": "user", "content": mensagem})
            agente["historico"].append({"role": "assistant", "content": texto})

            print(f"🤖 Modelo usado: {modelo}")
            return texto

        except Exception as e:
            erro = str(e)
            ultimo_erro = erro

            if "decommissioned" in erro:
                print(f"⚠️ Modelo descontinuado: {modelo}")
                continue
            else:
                print(f"⚠️ Erro com {modelo}: {erro}")
                continue

    raise RuntimeError(f"❌ Nenhum modelo funcionou.\nÚltimo erro: {ultimo_erro}")

# ==========================================
# SIMULAÇÃO COM INTERVENÇÃO
# ==========================================
def simular_conversa(turnos=10, tema_inicial="Problema na internet"):

    mensagem = tema_inicial
    log = []

    print("\n🎬 INÍCIO DA SIMULAÇÃO\n")

    for i in range(turnos):
        print(f"--- TURNO {i+1} ---\n")

        # ATENDENTE
        resp1 = gerar_resposta(mensagem, agente_1)
        print(f"🟢 {agente_1['nome']}: {resp1}\n")
        log.append(f"{agente_1['nome']}: {resp1}")

        time.sleep(1)

        # CLIENTE
        resp2 = gerar_resposta(resp1, agente_2)
        print(f"🔵 {agente_2['nome']}: {resp2}\n")
        log.append(f"{agente_2['nome']}: {resp2}")

        mensagem = resp2

        time.sleep(1)

        # INTERVENÇÃO DO USUÁRIO
        intervir = input("👉 Deseja interferir? (s/n): ").lower()

        if intervir == "s":
            msg_user = input("💬 Você entra na conversa: ")
            mensagem = msg_user
            log.append(f"VOCÊ: {msg_user}")

    print("\n🏁 FIM DA SIMULAÇÃO")

    # ==========================================
    # SALVAR LOG
    # ==========================================
    with open("conversa.txt", "w", encoding="utf-8") as f:
        for linha in log:
            f.write(linha + "\n")

    print("📄 Log salvo em conversa.txt")

# ==========================================
# EXECUÇÃO
# ==========================================
simular_conversa(
    turnos=8,
    tema_inicial="Estou sem internet há dias e quero cancelar imediatamente"
)

In [ ]:
# ==========================================
# INSTALAR DEPENDÊNCIA
# ==========================================
!pip install -q groq

# ==========================================
# IMPORTS
# ==========================================
from groq import Groq
from google.colab import userdata
import time

# ==========================================
# MODELOS (ATUALIZADOS + FALLBACK)
# ==========================================
MODELOS = [
    "llama-3.3-70b-versatile",
    "llama-3.3-8b-instant",
    "mixtral-8x7b-32768"
]

# ==========================================
# API KEY
# ==========================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("❌ API Key não encontrada nos Secrets")

client = Groq(api_key=GROQ_API_KEY)

print("✅ API conectada!")

# ==========================================
# CRIAR AGENTE DINÂMICO
# ==========================================
def criar_agente(nome, papel, estilo, objetivo, restricoes):
    prompt = f"""
    Nome: {nome}
    Papel: {papel}
    Estilo: {estilo}
    Objetivo: {objetivo}
    Restrições: {restricoes}

    Responda sempre de acordo com esse perfil.
    """

    return {
        "nome": nome,
        "prompt_base": prompt,
        "historico": []
    }

# ==========================================
# GERAR RESPOSTA (COM SUPERIOR INFLUENCIANDO)
# ==========================================
def gerar_resposta(mensagem, agente, contexto_superior=""):

    for modelo in MODELOS:
        try:
            resposta = client.chat.completions.create(
                model=modelo,
                messages=[
                    {
                        "role": "system",
                        "content": agente["prompt_base"] + "\n\n" + contexto_superior
                    },
                    *agente["historico"],
                    {"role": "user", "content": mensagem}
                ],
                temperature=0.7,
            )

            texto = resposta.choices[0].message.content

            agente["historico"].append({"role": "user", "content": mensagem})
            agente["historico"].append({"role": "assistant", "content": texto})

            print(f"🤖 {agente['nome']} usando: {modelo}")
            return texto

        except Exception as e:
            if "decommissioned" in str(e):
                continue
            else:
                continue

    raise RuntimeError("❌ Nenhum modelo disponível")

# ==========================================
# CONFIGURAÇÃO DINÂMICA DO CENÁRIO
# ==========================================
print("\n🎯 CONFIGURE O CENÁRIO\n")

cenario = input("Descreva o cenário: ")

ag1_nome = input("Nome do agente 1: ")
ag1_papel = input("Papel do agente 1: ")
ag1_estilo = input("Estilo (ex: calmo, agressivo): ")
ag1_obj = input("Objetivo: ")
ag1_rest = input("Restrições: ")

print("\n---\n")

ag2_nome = input("Nome do agente 2: ")
ag2_papel = input("Papel do agente 2: ")
ag2_estilo = input("Estilo: ")
ag2_obj = input("Objetivo: ")
ag2_rest = input("Restrições: ")

# Criar agentes
agente_1 = criar_agente(ag1_nome, ag1_papel, ag1_estilo, ag1_obj, ag1_rest)
agente_2 = criar_agente(ag2_nome, ag2_papel, ag2_estilo, ag2_obj, ag2_rest)

# ==========================================
# SIMULAÇÃO COM SUPERIOR
# ==========================================
def simular(turnos=10):

    mensagem = cenario
    contexto_superior = ""
    log = []

    print("\n🎬 INÍCIO DA SIMULAÇÃO\n")

    for i in range(turnos):
        print(f"\n--- TURNO {i+1} ---\n")

        # AGENTE 1
        resp1 = gerar_resposta(mensagem, agente_1, contexto_superior)
        print(f"🟢 {agente_1['nome']}: {resp1}\n")
        log.append(f"{agente_1['nome']}: {resp1}")

        time.sleep(1)

        # AGENTE 2
        resp2 = gerar_resposta(resp1, agente_2, contexto_superior)
        print(f"🔵 {agente_2['nome']}: {resp2}\n")
        log.append(f"{agente_2['nome']}: {resp2}")

        mensagem = resp2

        time.sleep(1)

        # ======================================
        # SUPERIOR INTERVÉM
        # ======================================
        acao = input("👔 Superior: (n=continua | i=intervir | m=mudar estratégia): ").lower()

        if acao == "i":
            instrucao = input("📢 Instrução do superior: ")
            contexto_superior += f"\nINSTRUÇÃO DO SUPERIOR: {instrucao}\n"
            log.append(f"SUPERIOR: {instrucao}")

        elif acao == "m":
            nova_estrategia = input("🔄 Nova diretriz geral: ")
            contexto_superior += f"\nNOVA ESTRATÉGIA: {nova_estrategia}\n"
            log.append(f"SUPERIOR (estratégia): {nova_estrategia}")

    print("\n🏁 FIM DA SIMULAÇÃO")

    with open("log_conversa.txt", "w", encoding="utf-8") as f:
        for linha in log:
            f.write(linha + "\n")

    print("📄 Log salvo!")

# ==========================================
# EXECUTAR
# ==========================================
simular(turnos=8)

In [ ]:
# ==========================================
# INSTALAR DEPENDÊNCIA
# ==========================================
!pip install -q groq

# ==========================================
# IMPORTS
# ==========================================
from groq import Groq
from google.colab import userdata
import time

# ==========================================
# MODELOS (ATUALIZADOS + FALLBACK)
# ==========================================
MODELOS = [
    "llama-3.3-70b-versatile",
    "llama-3.3-8b-instant",
    "mixtral-8x7b-32768"
]

# ==========================================
# API KEY
# ==========================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("❌ API Key não encontrada nos Secrets")

client = Groq(api_key=GROQ_API_KEY)

print("✅ API conectada!")

# ==========================================
# CRIAR AGENTE DINÂMICO
# ==========================================
def criar_agente(nome, papel, estilo, objetivo, restricoes):
    prompt = f"""
    Nome: {nome}
    Papel: {papel}
    Estilo: {estilo}
    Objetivo: {objetivo}
    Restrições: {restricoes}

    Responda sempre de acordo com esse perfil.
    """

    return {
        "nome": nome,
        "prompt_base": prompt,
        "historico": []
    }

# ==========================================
# GERAR RESPOSTA
# ==========================================
def gerar_resposta(mensagem, agente, contexto_superior=""):

    for modelo in MODELOS:
        try:
            resposta = client.chat.completions.create(
                model=modelo,
                messages=[
                    {
                        "role": "system",
                        "content": agente["prompt_base"] + "\n\n" + contexto_superior
                    },
                    *agente["historico"],
                    {"role": "user", "content": mensagem}
                ],
                temperature=0.7,
            )

            texto = resposta.choices[0].message.content

            agente["historico"].append({"role": "user", "content": mensagem})
            agente["historico"].append({"role": "assistant", "content": texto})

            print(f"🤖 {agente['nome']} usando: {modelo}")
            return texto

        except Exception as e:
            if "decommissioned" in str(e):
                print(f"⚠️ Modelo descontinuado: {modelo}")
                continue
            else:
                print(f"⚠️ Erro com {modelo}")
                continue

    raise RuntimeError("❌ Nenhum modelo disponível")

# ==========================================
# CONFIGURAÇÃO DO CENÁRIO
# ==========================================
print("\n🎯 CONFIGURE O CENÁRIO\n")

cenario = input("Descreva o cenário: ")

ag1_nome = input("Nome do agente 1: ")
ag1_papel = input("Papel do agente 1: ")
ag1_estilo = input("Estilo (ex: calmo, agressivo): ")
ag1_obj = input("Objetivo: ")
ag1_rest = input("Restrições: ")

print("\n---\n")

ag2_nome = input("Nome do agente 2: ")
ag2_papel = input("Papel do agente 2: ")
ag2_estilo = input("Estilo: ")
ag2_obj = input("Objetivo: ")
ag2_rest = input("Restrições: ")

# Criar agentes
agente_1 = criar_agente(ag1_nome, ag1_papel, ag1_estilo, ag1_obj, ag1_rest)
agente_2 = criar_agente(ag2_nome, ag2_papel, ag2_estilo, ag2_obj, ag2_rest)

# ==========================================
# SIMULAÇÃO COM CONTROLE TOTAL
# ==========================================
def simular(turnos=100):

    mensagem = cenario
    contexto_superior = ""
    log = []

    print("\n🎬 INÍCIO DA SIMULAÇÃO\n")

    for i in range(turnos):

        # PARADA ANTES DO TURNO
        parar = input("⏹ ENTER=continuar | s=parar: ").lower()
        if parar == "s":
            print("\n🛑 Simulação encerrada antes do turno!")
            break

        print(f"\n--- TURNO {i+1} ---\n")

        # AGENTE 1
        resp1 = gerar_resposta(mensagem, agente_1, contexto_superior)
        print(f"🟢 {agente_1['nome']}: {resp1}\n")
        log.append(f"{agente_1['nome']}: {resp1}")

        time.sleep(1)

        # AGENTE 2
        resp2 = gerar_resposta(resp1, agente_2, contexto_superior)
        print(f"🔵 {agente_2['nome']}: {resp2}\n")
        log.append(f"{agente_2['nome']}: {resp2}")

        mensagem = resp2

        time.sleep(1)

        # ======================================
        # SUPERIOR INTERVÉM
        # ======================================
        acao = input("👔 (n=continua | i=intervir | m=mudar estratégia | s=parar): ").lower()

        if acao == "s":
            print("\n🛑 Simulação interrompida pelo superior!")
            break

        elif acao == "i":
            instrucao = input("📢 Instrução do superior: ")
            contexto_superior += f"\nINSTRUÇÃO DO SUPERIOR: {instrucao}\n"
            log.append(f"SUPERIOR: {instrucao}")

        elif acao == "m":
            nova_estrategia = input("🔄 Nova diretriz geral: ")
            contexto_superior += f"\nNOVA ESTRATÉGIA: {nova_estrategia}\n"
            log.append(f"SUPERIOR (estratégia): {nova_estrategia}")

    print("\n🏁 FIM DA SIMULAÇÃO")

    # SALVAR LOG
    with open("log_conversa.txt", "w", encoding="utf-8") as f:
        for linha in log:
            f.write(linha + "\n")

    print("📄 Log salvo em log_conversa.txt")

# ==========================================
# EXECUTAR
# ==========================================
simular(turnos=50)

In [ ]:
# ==========================================
# INSTALAR DEPENDÊNCIA
# ==========================================
!pip install -q groq

# ==========================================
# IMPORTS
# ==========================================
from groq import Groq
from google.colab import userdata
import time

# ==========================================
# MODELOS (ATUALIZADOS + FALLBACK)
# ==========================================
MODELOS = [
    "llama-3.3-70b-versatile",
    "llama-3.3-8b-instant",
    "mixtral-8x7b-32768"
]

# ==========================================
# API KEY
# ==========================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("❌ API Key não encontrada nos Secrets")

client = Groq(api_key=GROQ_API_KEY)

print("✅ API conectada!")

# ==========================================
# CRIAR AGENTE DINÂMICO
# ==========================================
def criar_agente(nome, papel, estilo, objetivo, restricoes):
    prompt = f"""
    Nome: {nome}
    Papel: {papel}
    Estilo: {estilo}
    Objetivo: {objetivo}
    Restrições: {restricoes}

    Responda sempre de acordo com esse perfil.
    """

    return {
        "nome": nome,
        "prompt_base": prompt,
        "historico": []
    }

# ==========================================
# GERAR RESPOSTA COM TEMPO DE API
# ==========================================
def gerar_resposta(mensagem, agente, contexto_superior=""):

    for modelo in MODELOS:
        try:
            inicio = time.time()  # ⏱ início

            resposta = client.chat.completions.create(
                model=modelo,
                messages=[
                    {
                        "role": "system",
                        "content": agente["prompt_base"] + "\n\n" + contexto_superior
                    },
                    *agente["historico"],
                    {"role": "user", "content": mensagem}
                ],
                temperature=0.7,
            )

            fim = time.time()  # ⏱ fim
            tempo = round(fim - inicio, 2)

            texto = resposta.choices[0].message.content

            agente["historico"].append({"role": "user", "content": mensagem})
            agente["historico"].append({"role": "assistant", "content": texto})

            print(f"🤖 {agente['nome']} usando: {modelo} | ⏱ {tempo}s")
            return texto, tempo

        except Exception as e:
            if "decommissioned" in str(e):
                print(f"⚠️ Modelo descontinuado: {modelo}")
                continue
            else:
                print(f"⚠️ Erro com {modelo}")
                continue

    raise RuntimeError("❌ Nenhum modelo disponível")

# ==========================================
# CONFIGURAÇÃO DO CENÁRIO
# ==========================================
print("\n🎯 CONFIGURE O CENÁRIO\n")

cenario = input("Descreva o cenário: ")

ag1_nome = input("Nome do agente 1: ")
ag1_papel = input("Papel do agente 1: ")
ag1_estilo = input("Estilo (ex: calmo, agressivo): ")
ag1_obj = input("Objetivo: ")
ag1_rest = input("Restrições: ")

print("\n---\n")

ag2_nome = input("Nome do agente 2: ")
ag2_papel = input("Papel do agente 2: ")
ag2_estilo = input("Estilo: ")
ag2_obj = input("Objetivo: ")
ag2_rest = input("Restrições: ")

# Criar agentes
agente_1 = criar_agente(ag1_nome, ag1_papel, ag1_estilo, ag1_obj, ag1_rest)
agente_2 = criar_agente(ag2_nome, ag2_papel, ag2_estilo, ag2_obj, ag2_rest)

# ==========================================
# SIMULAÇÃO COM CONTROLE TOTAL
# ==========================================
def simular(turnos=100):

    mensagem = cenario
    contexto_superior = ""
    log = []

    print("\n🎬 INÍCIO DA SIMULAÇÃO\n")

    for i in range(turnos):

        # PARADA ANTES DO TURNO
        parar = input("⏹ ENTER=continuar | s=parar: ").lower()
        if parar == "s":
            print("\n🛑 Simulação encerrada antes do turno!")
            break

        print(f"\n--- TURNO {i+1} ---\n")

        # AGENTE 1
        resp1, t1 = gerar_resposta(mensagem, agente_1, contexto_superior)
        print(f"🟢 {agente_1['nome']} ({t1}s): {resp1}\n")
        log.append(f"{agente_1['nome']} ({t1}s): {resp1}")

        time.sleep(1)

        # AGENTE 2
        resp2, t2 = gerar_resposta(resp1, agente_2, contexto_superior)
        print(f"🔵 {agente_2['nome']} ({t2}s): {resp2}\n")
        log.append(f"{agente_2['nome']} ({t2}s): {resp2}")

        mensagem = resp2

        time.sleep(1)

        # ======================================
        # SUPERIOR INTERVÉM
        # ======================================
        acao = input("👔 (n=continua | i=intervir | m=mudar estratégia | s=parar): ").lower()

        if acao == "s":
            print("\n🛑 Simulação interrompida pelo superior!")
            break

        elif acao == "i":
            instrucao = input("📢 Instrução do superior: ")
            contexto_superior += f"\nINSTRUÇÃO DO SUPERIOR: {instrucao}\n"
            log.append(f"SUPERIOR: {instrucao}")

        elif acao == "m":
            nova_estrategia = input("🔄 Nova diretriz geral: ")
            contexto_superior += f"\nNOVA ESTRATÉGIA: {nova_estrategia}\n"
            log.append(f"SUPERIOR (estratégia): {nova_estrategia}")

    print("\n🏁 FIM DA SIMULAÇÃO")

    # SALVAR LOG
    with open("log_conversa.txt", "w", encoding="utf-8") as f:
        for linha in log:
            f.write(linha + "\n")

    print("📄 Log salvo em log_conversa.txt")

# ==========================================
# EXECUTAR
# ==========================================
simular(turnos=50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.4 MB/s eta 0:00:00
✅ API conectada!

🎯 CONFIGURE O CENÁRIO

Descreva o cenário: devedor e cobrador
Nome do agente 1: cobrador
Papel do agente 1: cobrar divida 
Estilo (ex: calmo, agressivo): calmo, experiente , conhece as leis , direitos e deveres dos clinete
Objetivo: receber a divida
Restrições: nao receber a divida

---

Nome do agente 2: devedor
Papel do agente 2: nao pagar a divida
Estilo: agressivo 
Objetivo: nao paga a divida
Restrições: nao pagar a vista

🎬 INÍCIO DA SIMULAÇÃO

⏹ ENTER=continuar | s=parar: 

--- TURNO 1 ---

🤖 cobrador usando: llama-3.3-70b-versatile | ⏱ 1.09s
🟢 cobrador (1.09s): Um encontro inevitável. Olá, sou o cobrador. Estou aqui para discutir a divida pendente que você possui conosco. Gostaria de saber se você está ciente do valor e das condições da divida.

Antes de começarmos, quero deixar claro que estou aqui para encontrar uma solução que atenda aos interesses de ambas as partes. É important